In [1]:
# Run this notebook with the "chuuchuu" conda environment (see environment.yml):
#   conda env create -f environment.yml   (first time only)
#   conda activate chuuchuu
#   python -m ipykernel install --user --name chuuchuu --display-name "Python (chuuchuu)"
# Then select the "Python (chuuchuu)" kernel for this notebook.
import sys
assert "chuuchuu" in sys.executable, "Wrong kernel selected: switch to 'Python (chuuchuu)' before running this notebook."

Aim of this script: Clean Chuuchuu dataset before analysis.
Especially: 
- Identify correct countries (using db ID)
- Identify stations

In [37]:
import pandas as pd
import numpy as np
import os

In [ ]:
#Which data use? 

data_selection = "combined"

# raw csv files live on the shared Google Drive, not in this repo
data_chuuchuu_folder = r"G:\.shortcut-targets-by-id\1fop_2EKRGMK369J15pyuC9039CxT_tnw\Rail_databases\punctuality_big_data\Chuuchuu"

if data_selection =="combined":
    data_chuuchuu_path = f"{data_chuuchuu_folder}/delay_records_test_combined_EU.csv"
elif data_selection =="june": data_chuuchuu_path = f"{data_chuuchuu_folder}/delay_records_2026-06-10_to_2026-06-17_EU.csv"
elif data_selection =="february": data_chuuchuu_path = f"{data_chuuchuu_folder}/delay_records_2026-02-26_to_2026-03-04_EU.csv"
elif data_selection =="french": data_chuuchuu_path = f"{data_chuuchuu_folder}/delay_records_2025_FR.csv"

In [39]:
# raw-import cache: skips re-parsing the (large, slow) source CSVs on reruns.
# only caches the raw import, not the cleaned result, since the cleaning steps below are still evolving.
cache_dir = "raw_data/cache"
os.makedirs(cache_dir, exist_ok=True)

stations_cache_path = f"{cache_dir}/data_stations.parquet"
osm_cache_path = f"{cache_dir}/data_osm_stations.parquet"
chuuchuu_cache_path = f"{cache_dir}/data_chuuchuu_{data_selection}.parquet"  # one cache per data_selection choice

if os.path.exists(stations_cache_path):
    data_stations = pd.read_parquet(stations_cache_path)
else:
    data_stations = pd.read_csv("sup_data/stations.csv", sep=";", low_memory=False)
    data_stations.to_parquet(stations_cache_path)

if os.path.exists(osm_cache_path):
    data_osm_stations = pd.read_parquet(osm_cache_path)
else:
    data_osm_stations = pd.read_csv("sup_data/EU_train_stations_OSM.csv", low_memory=False)
    data_osm_stations.to_parquet(osm_cache_path)

if os.path.exists(chuuchuu_cache_path):
    data_chuuchuu = pd.read_parquet(chuuchuu_cache_path)
else:
    data_chuuchuu = pd.read_csv(data_chuuchuu_path, low_memory=False)
    data_chuuchuu.to_parquet(chuuchuu_cache_path)

In [40]:
data_stations.head(5)

,id,name,slug,uic,uic8_sncf,latitude,longitude,parent_station_id,hub_id,country,...,info:ja,info:ko,info:pl,info:pt,info:ru,info:sv,info:tr,info:zh,normalised_code,iata_airport_code
0,1,Château-Arnoux – St-Auban,chateau-arnoux-st-auban,NaN,NaN,44.081790,6.001625,NaN,None,FR,...,None,None,None,None,None,None,None,None,urn:trainline:public:nloc:csv1,None
1,2,Château-Arnoux – St-Auban,chateau-arnoux-st-auban,8775123.0,87751230.0,44.061565,5.997373,1.0,None,FR,...,None,None,None,None,None,None,None,None,urn:trainline:public:nloc:csv2,None
2,3,Château-Arnoux Mairie,chateau-arnoux-mairie,8775122.0,87751222.0,44.063863,6.011248,1.0,None,FR,...,None,None,None,None,None,None,None,None,urn:trainline:public:nloc:csv3,None
3,4,Digne-les-Bains,digne-les-bains,NaN,NaN,44.350000,6.350000,NaN,None,FR,...,ディーニュ＝レ＝バン,디뉴레뱅,None,None,Динь-ле-Бен,None,None,迪涅萊班,urn:trainline:public:nloc:csv4,None
4,6,Digne-les-Bains,digne-les-bains,8775149.0,87751495.0,44.088710,6.222982,4.0,None,FR,...,ディーニュ＝レ＝バン,디뉴레뱅,None,None,Динь-ле-Бен,None,None,迪涅萊班,urn:trainline:public:nloc:csv6,None


In [41]:
data_chuuchuu.head()

,agency,routeType,routeNumber,date,deutscheBahnStopId,timestamp,originalRoute,originalStopId,stopName,arrival,...,plannedArrivalPlatform,arrivalCancelled,departure,plannedDeparture,departureDelay,departurePlatform,plannedDeparturePlatform,departureCancelled,extra,operator
0,NS,Sprinter,8148,2026-02-26,8400058,2026-02-25 04:51:47+00,Sprinter 8148,2992170,Amsterdam Centraal,2026-02-26 14:25:00+00,...,10a,f,None,2026-02-26 14:25:00+00,NaN,10a,10a,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": false, ""fu...",None
1,NS,Sprinter,8148,2026-02-26,8400059,2026-02-25 04:51:47+00,Sprinter 8148,2992236,Amsterdam Sloterdijk,2026-02-26 14:19:00+00,...,12,f,2026-02-26 14:19:00+00,2026-02-26 14:19:00+00,0.0,12,12,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": false, ""fu...",None
2,NS,Sprinter,8148,2026-02-26,8400561,2026-02-25 04:51:47+00,Sprinter 8148,3065389,Schiphol Airport,2026-02-26 14:07:00+00,...,3,f,2026-02-26 14:09:00+00,2026-02-26 14:09:00+00,0.0,3,3,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": false, ""fu...",None
3,NS,Sprinter,8148,2026-02-26,8400079,2026-02-25 04:51:47+00,Sprinter 8148,2860888,Amsterdam Lelylaan,2026-02-26 14:15:00+00,...,2,f,2026-02-26 14:15:00+00,2026-02-26 14:15:00+00,0.0,2,2,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": false, ""fu...",None
4,NS,Sprinter,8148,2026-02-26,8400332,2026-02-25 04:51:47+00,Sprinter 8148,2993022,Hoofddorp,None,...,3,f,2026-02-26 14:02:00+00,2026-02-26 14:02:00+00,0.0,3,3,f,"{""nsAgency"": ""IFF:NS"", ""unplanned"": false, ""fu...",None


### Identify countries using deutscheBahnStopId (UIC codes)

The first two digits of `deutscheBahnStopId` are the UIC country code. We map them to a country using `sup_data/UIC_country_codes.csv`.

If `deutscheBahnStopId` is longer than 8 characters, it is not a valid UIC-based stop id, so we flag it with an error value instead of guessing a country.

In [42]:
uic_codes = pd.read_csv("sup_data/UIC_country_codes.csv", sep=";")
uic_codes["Numerical code"] = uic_codes["Numerical code"].astype(str)
uic_codes.head()

,Numerical code,Alphabetical code,Country
0,10,FI,Finland
1,20,RU,Russia
2,21,BY,Belarus
3,22,UA,Ukraine
4,23,MD,Moldova


In [43]:
data_chuuchuu["deutscheBahnStopId"] = data_chuuchuu["deutscheBahnStopId"].astype(str)

# a valid UIC-based stop id is at most 8 characters long; anything longer signals a data issue
invalid_stop_id = data_chuuchuu["deutscheBahnStopId"].str.len() != 7
print(f"{invalid_stop_id.sum()} rows have a deutscheBahnStopId that is not exactly equal to 7 characters")

data_chuuchuu["uicCodeStop"] = data_chuuchuu["deutscheBahnStopId"].str[:2]

1237246 rows have a deutscheBahnStopId that is not exactly equal to 7 characters


In [44]:
data_chuuchuu = data_chuuchuu.merge(
    uic_codes[["Numerical code", "Country"]],
    how="left",                # keep all rows in data_chuuchuu
    left_on="uicCodeStop",     # column in data_chuuchuu
    right_on="Numerical code"  # column in uic_codes
)
data_chuuchuu = data_chuuchuu.drop(columns=["Numerical code"])
data_chuuchuu = data_chuuchuu.rename(columns={"Country": "country"})

# overwrite the country for stop ids we flagged as invalid, instead of trusting a coincidental match
data_chuuchuu.loc[invalid_stop_id, "country"] = "ERROR_INVALID_STOP_ID"

In [45]:
print(len(data_chuuchuu["country"].unique()))
data_chuuchuu["country"].value_counts(dropna=False)

26


country
Germany                  6861184
Italy                    1497934
ERROR_INVALID_STOP_ID    1237246
France                   1104083
Netherlands               835445
Denmark                   808944
Switzerland               763326
Hungary                   573669
Belgium                   557521
Austria                   472063
NaN                       156714
Poland                     57750
Sweden                     24433
Luxembourg                 23895
Czech Republic             10365
Slovenia                    5591
Romania                     4550
Slovakia                    1984
United Kingdom              1271
Croatia                     1070
Serbia                       720
Ukraine                      439
Spain                        358
Norway                        48
Iraq                           8
Portugal                       3
Name: count, dtype: int64

What are the agencies associated with stops unmatched? 

In [46]:
data_chuuchuu[data_chuuchuu["country"]=="ERROR_INVALID_STOP_ID"]["agency"].unique()

array(['GTFSDE', 'PL', 'NS', 'DB', 'OEBB', 'SBB'], dtype=object)

In [47]:
print(data_chuuchuu[data_chuuchuu["country"].isna()]["agency"].unique())
print(data_chuuchuu[data_chuuchuu["country"].isna()]["uicCodeStop"].unique())

['HU' 'OEBB']
['43' '36' '46']


In [48]:
print(data_chuuchuu[(data_chuuchuu["country"]=="ERROR_INVALID_STOP_ID")&(data_chuuchuu["agency"]=="GTFSDE")]["stopName"].unique())
print()
print(data_chuuchuu[(data_chuuchuu["country"]=="ERROR_INVALID_STOP_ID")&(data_chuuchuu["agency"]=="NS")]["stopName"].unique())
print()
print(data_chuuchuu[(data_chuuchuu["country"]=="ERROR_INVALID_STOP_ID")&(data_chuuchuu["agency"]=="DB")]["stopName"].unique())

['Rostock Hbf' 'Satzvey' 'Hauptbahnhof A1' 'Düsseldorf Flughafen Terminal'
 'Essen-Steele' 'Seefeld-Hechendorf' 'Hirschaid'
 'Arnstadt, Hauptbahnhof (1)' 'Sülzenbrücken' 'Erfurt Hbf' 'Anrath'
 'Köln-Buchforst' 'Übach-Palenberg' 'Rodgau-Hainhausen' 'Hückelhoven-Baal'
 'Langerwehe' 'Bruchsal' 'Köln-Stammheim'
 'Knielingen Eggensteiner Straße, Karlsruhe'
 'Knielingen Herweghstraße, Karlsruhe' 'Rheinhafen, Karlsruhe'
 'Darmstadt-Wixhausen' 'Mühlburg West, Karlsruhe'
 'Knielingen Siemens, Karlsruhe' 'Starckstraße, Karlsruhe' 'Köln West'
 'Linnich Bhf' 'Henstedt-Ulzburg' 'Oßmannstedt' 'Salzgitter-Ringelheim'
 'Voigtstedt' 'Wiesenburg(Sachs)' 'Fährbrücke' 'Ahlbeck Ostseetherme'
 'Schmiechen' 'Euskirchen-Stotzheim' 'Augartenstraße, Karlsruhe'
 'Kronenplatz (U), Karlsruhe' 'Ettlinger Tor/Staatstheater (U), Karlsruhe'
 'Kongresszentrum (U), Karlsruhe' 'Mühldorf(Oberbay)'
 'Erbprinz/Schloss, Ettlingen' 'August-Bebel-Straße, Karlsruhe'
 'Kurt-Schumacher-Straße, Karlsruhe'
 'Knielinger Allee/Städt.

### Fixing the remaining `ERROR_INVALID_STOP_ID` / NaN countries

We investigated the  `ERROR_INVALID_STOP_ID` rows and NaN rows and found they break down into a few well-defined, agency-specific patterns (verified by name-matching `stopName` against `sup_data/stations.csv`, which gave the same country nearly 100% of the time within each group):

- **`GTFSDE`, `NS`, `DB`** sometimes report a stop id with only 6 characters instead of the usual 7 (missing one digit) — every one of these that could be name-matched resolved to **Germany**.
- **`PL`** sometimes reports ids in other formats (5, 8, 9 or 10 characters, e.g. the composite `"44420_1_1"` style) — every one of these name-matched to **Poland**.
- **`HU`** uses two internal numbering prefixes, `36` (HÉV suburban lines) and `43` (GySEV/Raaberbahn regional lines), which aren't official UIC country codes — every stop under these prefixes name-matched to **Hungary**.

These rules only apply to rows currently marked `ERROR_INVALID_STOP_ID` or NaN — they never override a country already found via the UIC prefix match.

In [49]:
# rows we're still allowed to touch: only the ones without a country yet
needs_fix = data_chuuchuu["country"].isna() | (data_chuuchuu["country"] == "ERROR_INVALID_STOP_ID")
print(f"{needs_fix.sum()} rows still need a country before the rule-based fixes")

1393960 rows still need a country before the rule-based fixes


In [50]:
id_length = data_chuuchuu["deutscheBahnStopId"].str.len()

# GTFSDE / NS / DB report some stops with a 6-character id (missing the usual 7th digit) -> these are German stops
data_chuuchuu.loc[needs_fix & (data_chuuchuu["agency"] == "GTFSDE") & (id_length == 6), "country"] = "Germany"
data_chuuchuu.loc[needs_fix & (data_chuuchuu["agency"] == "NS") & (id_length == 6), "country"] = "Germany"
data_chuuchuu.loc[needs_fix & (data_chuuchuu["agency"] == "DB") & (id_length == 6), "country"] = "Germany"

# PL reports ids in non-standard formats (5, 8, 9 or 10 characters, e.g. "44420_1_1") -> these are Polish stops
data_chuuchuu.loc[needs_fix & (data_chuuchuu["agency"] == "PL") & (id_length != 7), "country"] = "Poland"

In [51]:
# HU numbers some stops with internal prefixes 36 (HEV suburban lines) and 43 (GySEV/Raaberbahn regional lines),
# which aren't official UIC country codes -> these are Hungarian stops
uic_prefix = data_chuuchuu["deutscheBahnStopId"].str[:2]
data_chuuchuu.loc[needs_fix & (data_chuuchuu["agency"] == "HU") & (uic_prefix.isin(["36", "43"])), "country"] = "Hungary"

still_missing = data_chuuchuu["country"].isna() | (data_chuuchuu["country"] == "ERROR_INVALID_STOP_ID")
print(f"{still_missing.sum()} rows still need a country after the rule-based fixes")

958559 rows still need a country after the rule-based fixes


### Fallback: name-based match against `stations.csv`

For any row still missing a country after the rules above, we fall back to matching `stopName` against `sup_data/stations.csv`'s `slug` column, which carries its own `country` field. This only ever fills in rows that are still `ERROR_INVALID_STOP_ID` or NaN — it never overrides a country already resolved.

`stations.csv` stores countries as 2-letter codes (e.g. `"DE"`), so we translate them through `uic_codes` to keep the `country` column consistently made of full country names.

In [52]:
stations_unique = data_stations.drop_duplicates(subset="slug", keep="first").copy()

def slugify(series):
    return (
        series.str.normalize("NFKD")     # split accented letters from their accents
        .str.encode("ascii", "ignore")   # drop the accents
        .str.decode("utf-8")
        .str.lower()
        .str.strip()
        .str.replace(r"\(", "-", regex=True)
        .str.replace(r"\)", "", regex=True)
        .str.replace(r"[^a-z0-9]+", "-", regex=True)  # collapse remaining punctuation/spaces
        .str.strip("-")
    )

data_chuuchuu["stopName_slug"] = slugify(data_chuuchuu["stopName"])

# translate stations.csv's 2-letter country codes into the full names used in our country column
alpha_to_country = uic_codes.drop_duplicates(subset="Alphabetical code", keep="first").set_index("Alphabetical code")["Country"]
stations_unique["country_full"] = stations_unique["country"].map(alpha_to_country)
slug_to_country = stations_unique.set_index("slug")["country_full"]

In [53]:
data_chuuchuu.loc[still_missing, "country"] = data_chuuchuu.loc[still_missing, "stopName_slug"].map(slug_to_country)

still_missing_after_fallback = data_chuuchuu["country"].isna() | (data_chuuchuu["country"] == "ERROR_INVALID_STOP_ID")
print(f"{still_missing_after_fallback.sum()} rows still need a country after the name-based fallback")

71554 rows still need a country after the name-based fallback


### Known exceptions

Cases where the id-based logic gives the wrong country (or no country at all) regardless of the rules above, each individually verified by web search and recorded in `sup_data/known_exceptions_country_match.csv`:

- **Basel Bad Bf** is physically in Switzerland, but is managed by DB and numbered as if it were a German station (`80` prefix) — should be **Switzerland**.
- **Charles de Gaulle Airport** has a `deutscheBahnStopId` whose `99` prefix coincidentally matches Iraq — should be **France**.
- **41 OEBB stops** (Vienna-area S-Bahn/bus stops, plus regional stops in Tyrol, Styria, Carinthia and Vorarlberg) that carry a platform/street annotation `stations.csv` doesn't have a matching slug for — should be **Austria**, except **Mittenwald**, a real town in Bavaria, Germany, even though it sits on an OEBB-served route through Tyrol.
- **PL**'s internal stop numbering doesn't always follow UIC conventions, and several of its ids coincidentally collide with ids that other agencies (GTFSDE, FR, SBB, NMBS) use as real UIC-prefixed ids for a completely different physical stop (e.g. `8012650` is PL's Pleszew *and* GTFSDE's Plessa). A handful of these PL stops (Olecko, Lipno, Chociwel, Babiak, Pleszew, Sarnaki, Żychlin, Niedźwiedź, Dorohusk, Baranówka, Gogolin, Tunel, Puck, Mieszkowice, Radymno, Warka, Parczew) are recorded here — should be **Poland**.
- **IT's Statte** name-matched to the Belgian town of Statte (near Huy, Wallonia) via the `stations.csv` fallback — should be **Italy**. This stop has no `deutscheBahnStopId` at all, so it can't be keyed the same way as the exceptions above (see below).

We join on **both `agency` and `deutscheBahnStopId` together**, not the id alone — otherwise a PL exception would leak onto an unrelated stop that just happens to share the same numeric id under a different agency. For exceptions with no `deutscheBahnStopId` (like Statte), keying on `agency` + a missing id would risk matching *every* other stop under that agency that's also missing an id, so those instead join on `agency` + `originalStopId`, which is unique per physical stop. Either way, the exception always wins regardless of what the automated rules produced for that stop.

In [54]:
known_exceptions = pd.read_csv(
    "sup_data/known_exceptions_country_match.csv",
    dtype={"deutscheBahnStopId": str, "originalStopId": str},
)

# join on (agency, deutscheBahnStopId) together, not just the id: the same numeric id can coincidentally
# belong to two unrelated physical stops under different agencies (e.g. PL's internal numbering happens
# to reuse ids that other agencies use as real UIC-prefixed ids), so an id-only join would leak one
# agency's exception onto a different, unrelated stop under another agency
id_exceptions = known_exceptions[known_exceptions["deutscheBahnStopId"].notna()]
exception_key = id_exceptions["agency"] + "|" + id_exceptions["deutscheBahnStopId"]
exception_country = pd.Series(id_exceptions["country"].values, index=exception_key)

data_key = data_chuuchuu["agency"] + "|" + data_chuuchuu["deutscheBahnStopId"]
data_chuuchuu["country"] = data_key.map(exception_country).fillna(data_chuuchuu["country"])

# exceptions with no deutscheBahnStopId (e.g. IT's Statte) can't be keyed the same way: since the id
# column is stringified to the literal "nan" for missing values, an id-based join would match every
# other stop under that agency that's also missing an id. These instead join on (agency, originalStopId),
# which is unique per physical stop
originalid_exceptions = known_exceptions[known_exceptions["originalStopId"].notna()]
exception_key_orig = originalid_exceptions["agency"] + "|" + originalid_exceptions["originalStopId"]
exception_country_orig = pd.Series(originalid_exceptions["country"].values, index=exception_key_orig)

data_key_orig = data_chuuchuu["agency"] + "|" + data_chuuchuu["originalStopId"].astype(str)
data_chuuchuu["country"] = data_key_orig.map(exception_country_orig).fillna(data_chuuchuu["country"])

### SBB stops: join against the official Swiss stop register (Didok)

SBB reports many stops using Swiss "sloid" identifiers (e.g. `ch:1:sloid:2204`) instead of a UIC-prefixed code — these are small/regional stops (often carrying a canton suffix like "ZH"/"SZ"/"FR") that aren't in `stations.csv`, so the name-based fallback above can't match them.

The Federal Office of Transport's official stop register (`sup_data/switzerland_stations_didok.csv`, see `switzerland_stations_didok.SOURCE.txt` for provenance) uses the exact same `sloid` identifiers and includes an ISO country code per stop, so we can join directly on the id instead of matching by name. This only touches SBB rows still unresolved after the fallback above.

In [55]:
swiss_stations = pd.read_csv("sup_data/switzerland_stations_didok.csv", sep=";", low_memory=False)

# Liechtenstein ("LI") has no UIC country code of its own, so it's missing from uic_codes -> add it manually
alpha_to_country_extra = pd.concat([alpha_to_country, pd.Series({"LI": "Liechtenstein"})])

sloid_to_country = (
    swiss_stations.drop_duplicates(subset="sloid")
    .set_index("sloid")["isocountrycode"]
    .map(alpha_to_country_extra)
)

sbb_still_missing = still_missing_after_fallback & (data_chuuchuu["agency"] == "SBB")
print(f"{sbb_still_missing.sum()} SBB rows targeted for the Didok join")

data_chuuchuu.loc[sbb_still_missing, "country"] = data_chuuchuu.loc[sbb_still_missing, "deutscheBahnStopId"].map(sloid_to_country)

# recompute so the final "unresolved" labeling below reflects what the Didok join just fixed
still_missing_after_fallback = data_chuuchuu["country"].isna() | (data_chuuchuu["country"] == "ERROR_INVALID_STOP_ID")
print(f"{still_missing_after_fallback.sum()} rows still need a country after the SBB Didok join")

68552 SBB rows targeted for the Didok join
880 rows still need a country after the SBB Didok join


### OEBB stops: retry the name match after stripping bus-platform annotations

Many OEBB stop names carry a platform/street annotation in parentheses, e.g. `"Bruck/Leitha Bahnhof (Bahnhofplatz)"`. The fallback's `slugify()` only replaces `(`/`)` with `-`/nothing — it doesn't remove the annotation text itself, so the slug (`"bruck-leitha-bahnhof-bahnhofplatz"`) never matches `stations.csv`'s plain `"bruck-leitha-bahnhof"`.

This retries the match for OEBB rows still unresolved, using a slug that strips the whole parenthetical suffix instead of just the brackets.

In [56]:
def slugify_stripped(series):
    return (
        series.str.normalize("NFKD")
        .str.encode("ascii", "ignore")
        .str.decode("utf-8")
        .str.lower()
        .str.strip()
        .str.replace(r"\(.*?\)", "", regex=True)   # drop the whole parenthetical suffix, not just the brackets
        .str.replace(r"[^a-z0-9]+", "-", regex=True)
        .str.strip("-")
    )

oebb_still_missing = still_missing_after_fallback & (data_chuuchuu["agency"] == "OEBB")
print(f"{oebb_still_missing.sum()} OEBB rows targeted for the retry")

retry_slug = slugify_stripped(data_chuuchuu.loc[oebb_still_missing, "stopName"])
data_chuuchuu.loc[oebb_still_missing, "country"] = retry_slug.map(slug_to_country)

still_missing_after_fallback = data_chuuchuu["country"].isna() | (data_chuuchuu["country"] == "ERROR_INVALID_STOP_ID")
print(f"{still_missing_after_fallback.sum()} rows still need a country after the OEBB retry")

880 OEBB rows targeted for the retry
0 rows still need a country after the OEBB retry


In [57]:
# anything left is genuinely unresolved -> label it explicitly instead of leaving NaN / the old error value
data_chuuchuu.loc[still_missing_after_fallback, "country"] = "UNKNOWN_COUNTRY"

In [58]:
print(len(data_chuuchuu["country"].unique()))
print()
display(data_chuuchuu["country"].value_counts(dropna=False))
print()
pct_unknown = len(data_chuuchuu[data_chuuchuu["country"] == "UNKNOWN_COUNTRY"]) / len(data_chuuchuu) * 100
print(f"{pct_unknown:.2f}% of rows have an unknown country")

21



country
Germany           7127845
Switzerland       1717414
Italy             1498924
France            1106153
Netherlands        835445
Denmark            808944
Hungary            730364
Belgium            557445
Austria            475458
Poland              67809
Sweden              24918
Luxembourg          23895
Czech Republic      10341
Slovenia             5591
Romania              4550
Slovakia             1982
Croatia              1064
United Kingdom        955
Serbia                720
Ukraine               439
Spain                 358
Name: count, dtype: int64


0.00% of rows have an unknown country


### Verification step: flag every agency's minority countries for review

For every agency, we treat its single most common country as its "home" country, and flag every other (agency, country) combination it has any rows under — regardless of size — as a candidate to manually verify, the same way we investigated SBB, OEBB and PL above. This won't auto-fix anything; it's a checklist. A flagged combination with a plausible geography (e.g. `DB`/Germany → Austria, a real neighboring country with heavy rail traffic) is likely fine as-is, while one with implausible geography or a tiny row count is worth drilling into with the same approach we used above (pull the unique stop names and verify by search), and any confirmed error should be added to `sup_data/known_exceptions.csv`.

In [59]:
agency_country_counts = (
    data_chuuchuu.groupby(["agency", "country"])
    .agg(rows=("deutscheBahnStopId", "size"), unique_stops=("deutscheBahnStopId", "nunique"))
    .reset_index()
)

# use unique stops (not rows) as the basis for the percentage: a single busy stop serviced hourly
# racks up far more rows than a real error affecting several rarely-serviced stops, so row counts
# don't reflect how many distinct stations are potentially mislabeled
agency_stop_totals = agency_country_counts.groupby("agency")["unique_stops"].transform("sum")
agency_country_counts["share_pct"] = (agency_country_counts["unique_stops"] / agency_stop_totals * 100).round(2)

# each agency's single most common country (by unique stops) is treated as its "home" country
home_country = agency_country_counts.loc[agency_country_counts.groupby("agency")["unique_stops"].idxmax(), ["agency", "country"]]
home_country = home_country.set_index("agency")["country"]
agency_country_counts["is_home_country"] = agency_country_counts["country"] == agency_country_counts["agency"].map(home_country)

flagged_summary = agency_country_counts[~agency_country_counts["is_home_country"]].sort_values(["agency", "unique_stops"], ascending=[True, False])
print(f"{len(flagged_summary)} (agency, country) combinations flagged for manual review (not the agency's home country)")
with pd.option_context("display.max_rows", None):
    display(flagged_summary.drop(columns=["is_home_country"]))

85 (agency, country) combinations flagged for manual review (not the agency's home country)


,agency,country,rows,unique_stops,share_pct
16,DB,Switzerland,53416,222,18.97
0,DB,Austria,72095,209,17.86
11,DB,Poland,6905,74,6.32
8,DB,Italy,3538,50,4.27
7,DB,Hungary,4174,41,3.50
14,DB,Slovenia,3150,36,3.08
3,DB,Czech Republic,5065,33,2.82
5,DB,France,1234,22,1.88
10,DB,Netherlands,2404,19,1.62
4,DB,Denmark,1697,17,1.45


Exporting the data as an intermediate output

In [60]:
export_data = input("Export intermediate data to parquet? (y/n): ")

if export_data.lower() == "y":
    intermediate_outputs_dir = "intermediate_outputs"
    os.makedirs(intermediate_outputs_dir, exist_ok=True)

    data_chuuchuu.to_parquet(f"{intermediate_outputs_dir}/data_chuuchuu_{data_selection}_countries.parquet")
else:''